In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
"""
RQ3 - Percepción de seguridad: script de reproducción del análisis reportado
en el paper (Sección RQ3), sobre el dataset filtrado final (n=59).

Exclusiones aplicadas sobre los 67 casos originales:
  1) Se descartan los respondientes que no contestaron NINGÚN ítem del
     bloque de seguridad (7 casos).
  2) Se excluye además el participante de la fila 54 de Excel, que no
     respondió la primera pregunta de seguridad (respuesta parcial que
     invalida el cálculo del score para ese caso).

Requiere: pandas, numpy, scipy, pingouin (pip install pingouin)
"""

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg

ARCHIVO_EXCEL = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"
COL_GRUPO = "Grupo"
COL_SUS_UX = "SUS UX"
COL_SUS_SEG = "seguridad sus"

# Fila de Excel a excluir (1-based, con encabezado) -> índice pandas
EXCEL_ROW_TO_EXCLUDE = 54
PANDAS_IDX_TO_EXCLUDE = EXCEL_ROW_TO_EXCLUDE - 2

df = repro_data.read_datos_cuanti()

# Los 6 ítems del bloque de seguridad, en el orden del cuestionario:
# odd (positivos): posiciones 0, 2, 4  |  even (negativos, reverse-coded): 1, 3, 5
SEC_ITEM_COLS = df.columns[13:19].tolist()

# --- Filtro: al menos un ítem respondido (n=60), luego excluir fila 54 (n=59) ---
respondio_algo = df[SEC_ITEM_COLS].notna().any(axis=1)
df59 = df[respondio_algo].drop(index=PANDAS_IDX_TO_EXCLUDE).copy()
print(f"n final: {len(df59)}")

# --- Cronbach's alpha (reverse-coding de los ítems negativos antes de calcular) ---
items_recoded = df59[SEC_ITEM_COLS].copy()
even_cols = [SEC_ITEM_COLS[1], SEC_ITEM_COLS[3], SEC_ITEM_COLS[5]]
for col in even_cols:
    items_recoded[col] = 6 - items_recoded[col]  # escala 1-5 invertida

alpha, ci_alpha = pg.cronbach_alpha(data=items_recoded)
print(f"Cronbach's alpha: {alpha:.3f}, IC95% {ci_alpha}")

# --- Grupos combinados: single-sig (T1+T4) vs multi-sig (T2+T3) ---
df59["scheme"] = df59[COL_GRUPO].map({1: "single", 4: "single", 2: "multi", 3: "multi"})
df59["security_score"] = (df59[COL_SUS_SEG] + 12) * 2.5  # score final 0-60

single = df59.loc[df59["scheme"] == "single", "security_score"]
multi = df59.loc[df59["scheme"] == "multi", "security_score"]

print(f"Single-sig (T1+T4), n={len(single)}: "
      f"M={single.mean():.2f}, SD={single.std(ddof=1):.2f}, Mdn={single.median():.2f}")
print(f"Multi-sig  (T2+T3), n={len(multi)}: "
      f"M={multi.mean():.2f}, SD={multi.std(ddof=1):.2f}, Mdn={multi.median():.2f}")

# --- Normalidad ---
print("Shapiro-Wilk single:", stats.shapiro(single))
print("Shapiro-Wilk multi:", stats.shapiro(multi))

# --- Welch's t-test ---
n1, n2 = len(single), len(multi)
v1, v2 = single.var(ddof=1), multi.var(ddof=1)
t_res = stats.ttest_ind(single, multi, equal_var=False)
df_welch = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
print(f"Welch t-test: t={t_res.statistic:.3f}, df={df_welch:.2f}, p={t_res.pvalue:.4f}")

# --- Mann-Whitney U ---
mw = stats.mannwhitneyu(single, multi, alternative="two-sided")
print(f"Mann-Whitney U={mw.statistic}, p={mw.pvalue:.4f}")

# --- Diferencia de medias, IC 95%, Hedges' g ---
diff = single.mean() - multi.mean()
se = np.sqrt(v1 / n1 + v2 / n2)
ci = stats.t.interval(0.95, df_welch, loc=diff, scale=se)
print(f"Diferencia de medias: {diff:.2f}, IC95%: [{ci[0]:.2f}, {ci[1]:.2f}]")

sp = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
d = diff / sp
J = 1 - (3 / (4 * (n1 + n2) - 9))  # corrección de Hedges
g = d * J
print(f"Cohen's d: {d:.3f}, Hedges' g: {g:.3f}")

# --- Atrición: ¿relacionada con el tratamiento o con el SUS UX? ---
excluidos_idx = df[df[SEC_ITEM_COLS].isna().all(axis=1)].index.tolist() + [PANDAS_IDX_TO_EXCLUDE]
df["excluido"] = df.index.isin(excluidos_idx)

tabla = pd.crosstab(df[COL_GRUPO], df["excluido"])
chi2, p_chi2, dof, exp = stats.chi2_contingency(tabla)
print(f"Chi2 (atrición vs. tratamiento): chi2={chi2:.3f}, df={dof}, p={p_chi2:.3f}")

sus_excl = df.loc[df["excluido"], COL_SUS_UX]
sus_incl = df.loc[~df["excluido"], COL_SUS_UX]
mw_atricion = stats.mannwhitneyu(sus_excl, sus_incl, alternative="two-sided")
print(f"Mann-Whitney (atrición vs. SUS UX): U={mw_atricion.statistic}, p={mw_atricion.pvalue:.3f}")

# --- TOST (test de equivalencia), márgenes en unidades de Cohen's d ---
for d_bound in [0.3, 0.5, 0.8]:
    raw_bound = d_bound * sp
    tost = pg.tost(single, multi, bound=raw_bound, paired=False, correction=True)
    p_tost = tost["pval"].values[0]
    print(f"TOST con margen d={d_bound} (±{raw_bound:.2f} puntos): p={p_tost:.4f}")

n final: 59
Cronbach's alpha: 0.780, IC95% [0.679 0.856]
Single-sig (T1+T4), n=28: M=42.41, SD=11.02, Mdn=42.50
Multi-sig  (T2+T3), n=31: M=40.73, SD=13.65, Mdn=42.50
Shapiro-Wilk single: ShapiroResult(statistic=np.float64(0.9589898090189538), pvalue=np.float64(0.32987702351759596))
Shapiro-Wilk multi: ShapiroResult(statistic=np.float64(0.956251896082683), pvalue=np.float64(0.23152872798132818))
Welch t-test: t=0.524, df=56.33, p=0.6025
Mann-Whitney U=459.5, p=0.7032
Diferencia de medias: 1.68, IC95%: [-4.76, 8.13]
Cohen's d: 0.135, Hedges' g: 0.133
Chi2 (atrición vs. tratamiento): chi2=5.535, df=3, p=0.137
Mann-Whitney (atrición vs. SUS UX): U=238.0, p=0.977
TOST con margen d=0.3 (±3.74 puntos): p=0.2625
TOST con margen d=0.5 (±6.24 puntos): p=0.0813
TOST con margen d=0.8 (±9.98 puntos): p=0.0063
